# 01 · Generate and validate synthetic transfers

**Question:** Can we construct a reproducible referral cohort with explicit provenance, observable note labels, and validation? Entirely fictional; no real patient source.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
get_ipython().run_line_magic('matplotlib', 'inline')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
DATA = ROOT / 'data/synthetic/transfer_requests.csv'
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.spines.top': False, 'axes.spines.right': False})


## Scenario design
Sixteen authored adult scenarios vary service, respiratory support, pressors, monitoring, and specialty needs. The target is assigned before prose rendering. It is a simulation assumption, not a clinical label. Eight prose families partition 625/125/250 records into train/validation/test.

In [2]:
from transfer_assistant.synthetic import generate_dataset, validate_dataset, write_dataset, SEED
df = write_dataset(ROOT)
validation = validate_dataset(df)
assert df.equals(generate_dataset(seed=SEED))
display(validation)

{'passed': True,
 'checks': {'row_count': True,
  'unique_ids': True,
  'unique_notes': True,
  'all_synthetic': True,
  'age_range': True,
  'vital_ranges': True,
  'required_columns': True,
  'all_routes': True,
  'fictional_facilities': True,
  'synthetic_note_marker': True,
  'no_template_overlap': True,
  'complete_source_truth': True,
  'annotation_keys': True,
  'annotation_lineage': True,
  'routes_in_each_split': True},
 'rows': 1000,
 'seed': 20260909,
 'class_counts': {'ICU': 303,
  'telemetry': 251,
  'medical/surgical': 244,
  'specialty review': 202},
 'split_counts': {'train': 625, 'test': 250, 'validation': 125},
 'records_with_omissions': 685,
 'records_with_conflicts': 39}

In [3]:
display(df[['request_id','patient_age','presenting_problem','requested_service','oxygen_support','vasopressor_use','routing_target','split']].head(8))
display(pd.crosstab(df.split, df.routing_target))

,request_id,patient_age,presenting_problem,requested_service,oxygen_support,vasopressor_use,routing_target,split
0,SYN-0001,78,hypoxemic pneumonia,pulmonology,high-flow nasal cannula,False,ICU,train
1,SYN-0002,30,dehydration,internal medicine,room air,False,medical/surgical,train
2,SYN-0003,89,syncope,internal medicine,room air,False,telemetry,train
3,SYN-0004,80,respiratory failure,pulmonology,mechanical ventilation,False,ICU,train
4,SYN-0005,82,complex fracture,orthopedics,room air,False,specialty review,train
5,SYN-0006,39,pneumonia,internal medicine,nasal cannula,False,medical/surgical,validation
6,SYN-0007,30,heart failure,cardiology,nasal cannula,False,telemetry,test
7,SYN-0008,43,urinary tract infection,internal medicine,room air,False,medical/surgical,test


routing_target,ICU,medical/surgical,specialty review,telemetry
split,,,,
test,82,52,45,71
train,191,156,128,150
validation,30,36,29,30


## Complete scenario versus referral evidence
A negative is observable information; a missing value is not. Conflicts are separately recorded but unavailable to routing until clarified. Compare one incomplete referral's source truth with its visible annotation.

In [4]:
row = df[df.omitted_fields.ne('[]')].iloc[0]
print(row.free_text_transfer_note)
visible = json.loads(row.visible_annotations)
display(pd.DataFrame([{'field': k, 'source_truth': str(row[k]), 'note_visible': str(v)} for k,v in visible.items()]))

SYNTHETIC TRAINING CASE. Vasopressor use: no. Requested level of care: ICU. SpO2 95%. Oxygen support: high-flow nasal cannula. HR 132 bpm. Continuous cardiac monitoring required. Consultants: pulmonology. Presenting problem: hypoxemic pneumonia. Requested service: pulmonology. SBP 155 mmHg. Age: 78. Referring facility: Synthetic North Hospital. Urgency: emergent. No specialty evaluation required. Bed availability: available.


,field,source_truth,note_visible
0,bed_availability,available,available
1,heart_rate,132,132
2,isolation_requirements,none,None
3,monitoring_required,True,True
4,oxygen_support,high-flow nasal cannula,high-flow nasal cannula
5,patient_age,78,78
6,presenting_problem,hypoxemic pneumonia,hypoxemic pneumonia
7,referring_facility,Synthetic North Hospital,Synthetic North Hospital
8,relevant_consultants,pulmonology,pulmonology
9,requested_level_of_care,ICU,ICU


In [5]:
family_sets = {s: set(g.template_family) for s,g in df.groupby('split')}
assert not family_sets['train'] & family_sets['test']
assert not family_sets['validation'] & family_sets['test']
display(family_sets)

{'test': {6, 7}, 'train': {0, 1, 2, 3, 4}, 'validation': {5}}

## Interpretation
These checks establish consistency and lineage, not clinical realism. Documentation omission rates and physiological ranges are authored assumptions. Never use hidden source truth to fill inference-time gaps. See the data dictionary for every field and generation parameter.